# Preparations

## Installation

In [ ]:
# https://docs.unsloth.ai/get-started/installing-+-updating/google-colab
# import os
# if "COLAB_" in "".join(os.environ.keys()):
#     # Do this only in Colab notebooks! Otherwise use pip install unsloth
#     !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#     !pip install --nodeps xformers "trl<0.9.0" peft accelerate bitsandbytes

## Helper functions for JSON handling

In [ ]:
def unescape(input_str: str | None) -> str | None:
    if input_str is None:
        return None

    result = []
    i = 0
    while i < len(input_str):
        c = input_str[i]
        if c != '\\':
            result.append(c)
            i += 1
            continue

        if i + 1 >= len(input_str):
            raise ValueError(f"Trailing backslash at index {i}")

        next_c = input_str[i + 1]
        if next_c == '"':    result.append('"');  i += 2
        elif next_c == '\\': result.append('\\'); i += 2
        elif next_c == '/':  result.append('/');  i += 2
        elif next_c == 'b':  result.append('\b'); i += 2
        elif next_c == 'f':  result.append('\f'); i += 2
        elif next_c == 'n':  result.append('\n'); i += 2
        elif next_c == 'r':  result.append('\r'); i += 2
        elif next_c == 't':  result.append('\t'); i += 2
        elif next_c == 'u':
            if i + 5 >= len(input_str):
                raise ValueError(f"Incomplete \\uXXXX at index {i}")
            result.append(chr(int(input_str[i + 2:i + 6], 16)))
            i += 6
        else:
            raise ValueError(f"Invalid escape '\\{next_c}' at index {i}")

    return ''.join(result)


def escape(input_str: str | None) -> str:
    if input_str is None:
        return "null"

    result = []
    for c in input_str:
        if   c == '"':  result.append('\\"')
        elif c == '\\': result.append('\\\\')
        elif c == '\b': result.append('\\b')
        elif c == '\f': result.append('\\f')
        elif c == '\n': result.append('\\n')
        elif c == '\r': result.append('\\r')
        elif c == '\t': result.append('\\t')
        elif ord(c) < 0x20:
            result.append(f'\\u{ord(c):04x}')
        else:
            result.append(c)

    return ''.join(result)

## Check if CUDA is available

In [ ]:
import torch

if torch.cuda.is_available():
  print("CUDA is available")
  print(torch.cuda.get_device_name(0))
else:
  print("CUDA is not available")


## Load the model for inference

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3",
    # model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    # model_name = "unsloth/phi-3.5-mini-instruct",
    # model_name = "unsloth/tinyllama-chat",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [ ]:
FastLanguageModel.for_inference(model);

# Try it with a single example

In [ ]:
system = '''
Re-phrase the text below in whole sentences. Write continuous text and avoid enumerations
or numbered paragraphs. Pay attention to not forget any information. Make the word count roughly match the size of the input text:
'''

In [ ]:
input = """
// The following sentence describes the configuration of a timelapse image acquisition workflow, which 
// is repeated at an interval of 10 minute(s) for a duration of 4 hour(s).
// Acquisition starts at a specific time, at 10:00.
// All the positions defined above are imaged.
// All the channels defined above are imaged.
// The plane distance, i.e. the z-step is 30.1 microns,
// the objective lens used is the 20x lens,
// in combination with the 0.5x magnification changer.
// Camera binning is set to 1 x 1 pixels (no binning).
At 10:00, acquire...
  every 10 minute(s) for 4 hour(s)
  all positions
  all channels
  with a plane distance of 30.1 microns
  using the 20x lens with the 0.5x magnification changer and a binning of 1 x 1.
"""

In [ ]:
chat = [
    {"role": "system", "content": system},
    {"role": "user", "content": input}]

sample = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt = True)
# sample = prompt.format(system, input, "")
print(sample)



In [ ]:
%%time

# Use case             top_p       top_k     Description
# -----------------------------------------------------------------------------------
# Creative writing     0.92–0.97   50–100    Allows diverse, expressive outputs
# Conversational/chat  0.85–0.95   50–80     Keeps responses interesting but on-topic
# Safer, focused gen   0.8         20–50     Reduces drift or hallucination
# Highly controlled    0.6–0.8     10–20     Close to deterministic, very focused

# temperature: Lower  (e.g.       0.5) = safer, more conservative
#              Higher (e.g. 1.0 - 1.3) = more creative, more chaotic

nSamples = 1
inputs = tokenizer(sample, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, top_p=0.97, top_k=100, temperature=1.2, do_sample=True, num_return_sequences=nSamples)
# outputs = model.generate(**inputs, max_new_tokens=4096, use_cache=True, top_p=0.9, top_k=80, temperature=0.5, do_sample=True, num_return_sequences=nSamples)

for o in range(nSamples):
    full_output = tokenizer.decode(outputs[o], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    generated_text = full_output[len(prompt_text):].strip()
    print(generated_text)

# Apply it to an entire dataset

## Load the dataset

In [ ]:
from datasets import load_dataset

# For standard JSON formcat
dataset = load_dataset("json", data_files="autogenerated-sentences-with-context.json")

print(dataset)

## Print the first sample

In [ ]:
print(dataset['train'][0]['context'])

In [ ]:
print(dataset['train'][0]['sentence'])

In [ ]:
print(dataset['train'][0]['json'])

## Test for the first entry

In [ ]:
    l = dataset['train'][0]
    sentence = l['sentence']
    context = l['context']

    chat = [
        {"role": "system", "content": system},
        {"role": "user", "content": "Script:\n" + sentence + "\nDescription:\n"},
    ]
    
    sample = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt = True)
    inputs = tokenizer(sample, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=False, top_p=0.97, top_k=100, temperature=1, do_sample=True, num_return_sequences=1)

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    response = full_output[len(prompt_text):].strip()
    print(sample)
    print(response)

## Apply to the entire dataset

In [ ]:
import re
import time
import os

if not os.path.exists('sentence-samples'):
    os.makedirs('sentence-samples')

# for i in range(1):
for i in range(0, len(dataset['train'])):
    l = dataset['train'][i]
    sentence = l['sentence']
    context = l['context']
    json = l['json']

    chat = [
        {"role": "system", "content": system},
        {"role": "user", "content": "Script:\n" + sentence + "\nDescription:\n"},
    ]
    
    start = time.time()
    
    sample = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt = True)
    inputs = tokenizer(sample, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True, top_p=0.97, top_k=100, temperature=1, do_sample=True, num_return_sequences=1)

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    response = full_output[len(prompt_text):].strip()

    duration = time.time() - start
    print(f'Iteration {i}: {duration:.2f}')

    fi = open(f'sentence-samples/{i:04}-original.txt', 'w')
    fi.write(sentence)
    fi.close()

    fi = open(f'sentence-samples/{i:04}-rephrased.txt', 'w')
    fi.write(response)
    fi.close()

    fi = open(f'sentence-samples/{i:04}-context.txt', 'w')
    fi.write(context)
    fi.close()

    fi = open(f'sentence-samples/{i:04}-json.txt', 'w')
    fi.write(json)
    fi.close()



## Split the samples in training and test data

In [ ]:
import os

N = 2000  # len(dataset['train'])
nTrain = int(0.8 * N)
nTest = N - nTrain

print("Split into", nTrain, "training samples and", nTest, "test samples") 

### Create the train dataset

In [ ]:
import re

f = open("dataset-for-finetuning-sentences-train.json", "w")
f.write("")

for i in range(0, nTrain):
    fi = open(f'sentence-samples/{i:04}-original.txt', 'r')
    original = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-rephrased.txt', 'r')
    rephrased = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-context.txt', 'r')
    context = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-json.txt', 'r')
    json = fi.read()
    fi.close()

    f.write('{"sentence": "' + escape(original) + '", "rephrased": "' + escape(rephrased) + '", "context": "' + escape(context) + '", "json": "' + escape(json) + '"}\n')
    f.flush()

f.close()

### Create the test dataset

In [ ]:
f = open("dataset-for-finetuning-sentences-test.json", "w")
f.write("")

for i in range(nTrain, N):
    fi = open(f'sentence-samples/{i:04}-original.txt', 'r')
    original = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-rephrased.txt', 'r')
    rephrased = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-context.txt', 'r')
    context = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-json.txt', 'r')
    json = fi.read()
    fi.close()

    f.write('{"sentence": "' + escape(original) + '", "rephrased": "' + escape(rephrased) + '", "context": "' + escape(context) + '", "json": "' + escape(json) + '"}\n')
    f.flush()

f.close()